# Version 1

In [ ]:
import numpy as np

class LightningImpulseAnalyzer:
    def __init__(self, time_array, voltage_array):
        self.raw_time = np.array(time_array)
        self.raw_voltage = np.array(voltage_array)
        
        # Aquí guardaremos la señal con el cero corregido
        self.zeroed_voltage = None
        self.offset_value = 0.0

        # Validaciones
        if len(self.raw_time) != len(self.raw_voltage):
            raise ValueError("Los arrays de tiempo y voltaje deben tener la misma longitud.")

    def remove_offset(self, pre_trigger_percent=10):
        """
        Calcula el promedio de la señal en la zona de pre-disparo (antes del impulso)
        y lo resta a toda la señal para asegurar que el voltaje base sea 0V.
        
        Args:
            pre_trigger_percent (float): Porcentaje inicial de la muestra que se 
                                         considera 'silencio' o ruido de fondo. 
                                         Por defecto 10%.
        """
        # 1. Determinar cuántas muestras corresponden al pre-trigger
        total_samples = len(self.raw_voltage)
        n_samples_background = int(total_samples * (pre_trigger_percent / 100))
        
        if n_samples_background < 1:
            raise ValueError("No hay suficientes muestras para calcular el offset.")

        # 2. Calcular el promedio en esa zona (Nivel de ruido base)
        # Tomamos los primeros datos donde se supone que aún no hay impulso
        background_noise = self.raw_voltage[:n_samples_background]
        self.offset_value = np.mean(background_noise)
        
        # 3. Restar ese valor a toda la señal
        self.zeroed_voltage = self.raw_voltage - self.offset_value
        
        return self.offset_value

 # Version 2

In [ ]:
import numpy as np

class LightningImpulseAnalyzer:
    def __init__(self, voltage_array, sample_interval):
        """
        Inicializa el analizador.
        
        Args:
            voltage_array (list/array): Datos crudos de voltaje del osciloscopio.
            sample_interval (float): El paso de tiempo entre muestras (ej. 10e-9 para 10ns).
        """
        self.raw_voltage = np.array(voltage_array)
        self.ts = sample_interval  # Tasa de muestreo (Sampling Interval)
        
        # Generamos el array de tiempo automáticamente: t = index * intervalo
        self.raw_time = np.arange(len(self.raw_voltage)) * self.ts
        
        # Variables de estado para el proceso
        self.zeroed_voltage = None
        self.offset_value = 0.0

    def remove_offset(self, pre_trigger_percent=10):
        """
        Etapa 1: Corrección de línea base.
        Calcula el promedio en la zona de pre-disparo y lo resta.
        """
        # 1. Calcular cuántas muestras son ruido de fondo
        total_samples = len(self.raw_voltage)
        n_samples_background = int(total_samples * (pre_trigger_percent / 100))
        
        if n_samples_background < 1:
            raise ValueError("No hay suficientes muestras de pre-trigger para calcular el offset.")

        # 2. Calcular el promedio del ruido (Offset)
        # Tomamos el slice desde 0 hasta el porcentaje indicado
        background_noise = self.raw_voltage[:n_samples_background]
        self.offset_value = np.mean(background_noise)
        
        # 3. Restar el offset a toda la señal
        self.zeroed_voltage = self.raw_voltage - self.offset_value
        
        return self.offset_value